# GEE H3 features

H3 v7 country spine and GEE feature blocks:

- AlphaEarth polygon means
- VIIRS 24-month signature
- MODIS 16-day NDVI summary
- GHSL building height
- Google Open Buildings pull and morphology

Accessibility, WorldPop and OSM stay out of this stage. final merge uses the local feature tables.

In [1]:
!pip -q install earthengine-api geemap geopandas pyogrio shapely h3 pyarrow pandas numpy scipy scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 84.9 MB/s eta 0:00:00


In [2]:
from __future__ import annotations

import json
import math
import os
import time
from datetime import datetime
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import h3
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import shapely
from google.colab import drive
from scipy.spatial import cKDTree
from shapely.geometry import Point, Polygon, MultiPolygon
from tqdm.auto import tqdm

ee.Authenticate()
ee.Initialize(project="socioeconomic-embeddings")
drive.mount(str(Path("drive").resolve()))

pd.set_option("display.max_columns", 200)

Mounted at /content/drive


In [3]:
# config

DRIVE_ROOT = Path("drive") / "MyDrive"
OUTPUT_ROOT = DRIVE_ROOT / "h3_tabular_v2"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

H3_RES = 7
BATCH_SIZE = 1000

# source year caps; no partial future year
SOURCE_LATEST_YEAR = {
    "alphaearth": 2024,       # collection ends 2024
    "viirs_full_year": 2025,  # last complete year used
    "sentinel2_full_year": 2025,
    "ghsl_height": 2018,
    "gob_inference": 2023,
}

COUNTRY_CONFIG = {
    "NGA": {"name": "Nigeria", "gaul_name": "Nigeria", "reference_year": 2024, "survey_label": "DHS 2024"},
    "TZA": {"name": "Tanzania", "gaul_name": "United Republic of Tanzania", "reference_year": 2024, "survey_label": "assumed_2024"},
    "ETH": {"name": "Ethiopia", "gaul_name": "Ethiopia", "reference_year": 2025, "survey_label": "DHS 2024-25"},
    "GMB": {"name": "Gambia", "gaul_name": "Gambia", "reference_year": 2020, "survey_label": "DHS 2019-20"},
    "GHA": {"name": "Ghana", "gaul_name": "Ghana", "reference_year": 2022, "survey_label": "DHS 2022"},
    "KEN": {"name": "Kenya", "gaul_name": "Kenya", "reference_year": 2025, "survey_label": "Interim DHS 2025"},
    "SLE": {"name": "Sierra Leone", "gaul_name": "Sierra Leone", "reference_year": 2026, "survey_label": "DHS 2026"},
    "TGO": {"name": "Togo", "gaul_name": "Togo", "reference_year": 2026, "survey_label": "DHS 2026"},
    "ZMB": {"name": "Zambia", "gaul_name": "Zambia", "reference_year": 2024, "survey_label": "DHS 2024"},
}

COUNTRIES = list(COUNTRY_CONFIG)

GEE_OUT = OUTPUT_ROOT / "gee_outputs"
GEE_OUT.mkdir(exist_ok=True)

CELLS_PATH = OUTPUT_ROOT / f"h3_res{H3_RES}_country_spine.parquet"
ALPHAEARTH_PATH = GEE_OUT / "alphaearth_h3_mean.parquet"
VIIRS_PATH = GEE_OUT / "viirs_nightlights_24m_unfloored.parquet"
NDVI_PATH = GEE_OUT / "modis_ndvi_16d_annual.parquet"
GHSL_HEIGHT_PATH = GEE_OUT / "ghsl_building_height.parquet"
GEE_FEATURES_PATH = GEE_OUT / "gee_h3_features.parquet"
GOB_MORPH_PATH = GEE_OUT / "gob_h3_morphology.parquet"
FINAL_PATH = OUTPUT_ROOT / "cells_full_tabular_v2.parquet"

print(f"Output root: {OUTPUT_ROOT}")

Output root: h3_tabular_v2


## H3 spine

Full-country H3 v7 cells from GAUL admin 0. admin1 assigned by centroid.

In [4]:
def fetch_gaul_country_boundaries() -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame]:
    level0 = ee.FeatureCollection("FAO/GAUL/2015/level0")
    level1 = ee.FeatureCollection("FAO/GAUL/2015/level1")

    adm0_frames = []
    adm1_frames = []
    for iso, cfg in COUNTRY_CONFIG.items():
        name = cfg["gaul_name"]
        print(f"Fetching GAUL boundaries for {iso}: {name}")
        fc0 = level0.filter(ee.Filter.eq("ADM0_NAME", name))
        fc1 = level1.filter(ee.Filter.eq("ADM0_NAME", name))
        g0 = geemap.ee_to_gdf(fc0)
        g1 = geemap.ee_to_gdf(fc1)
        if g0.empty:
            raise ValueError(f"No GAUL level0 boundary found for {iso} / {name}")
        g0["country_iso3"] = iso
        g1["country_iso3"] = iso
        adm0_frames.append(g0[["country_iso3", "ADM0_NAME", "geometry"]])
        adm1_frames.append(g1[["country_iso3", "ADM1_NAME", "geometry"]])

    adm0 = gpd.GeoDataFrame(pd.concat(adm0_frames, ignore_index=True), crs="EPSG:4326")
    adm1 = gpd.GeoDataFrame(pd.concat(adm1_frames, ignore_index=True), crs="EPSG:4326")
    return adm0, adm1


from shapely.geometry import Polygon, MultiPolygon, GeometryCollection

def iter_polygons(geom):
    if geom is None or geom.is_empty:
        return

    if isinstance(geom, Polygon):
        yield geom

    elif isinstance(geom, MultiPolygon):
        for part in geom.geoms:
            yield from iter_polygons(part)

    elif isinstance(geom, GeometryCollection):
        for part in geom.geoms:
            yield from iter_polygons(part)

    else:
        # skip non-area parts
        return


def polygon_to_h3_cells(geom, res: int) -> set[str]:
    out = set()

    for poly in iter_polygons(geom):
        exterior = [(lat, lon) for lon, lat in poly.exterior.coords]
        holes = [
            [(lat, lon) for lon, lat in ring.coords]
            for ring in poly.interiors
        ]

        shape = h3.LatLngPoly(exterior, *holes)
        out.update(h3.polygon_to_cells(shape, res))

    return out


def h3_boundary_polygon(cell: str) -> Polygon:
    boundary = h3.cell_to_boundary(cell)
    return Polygon([(lon, lat) for lat, lon in boundary])


def build_spine() -> gpd.GeoDataFrame:
    adm0, adm1 = fetch_gaul_country_boundaries()
    rows = []

    for iso in COUNTRIES:
        geom = adm0.loc[adm0["country_iso3"] == iso, "geometry"].unary_union
        cells = sorted(polygon_to_h3_cells(geom, H3_RES))
        print(f"{iso}: {len(cells):,} H3 cells")

        for cell in tqdm(cells, desc=f"{iso} h3 rows"):
            lat, lon = h3.cell_to_latlng(cell)
            rows.append({
                "country_iso3": iso,
                "h3_index": cell,
                "centroid_lat": lat,
                "centroid_lon": lon,
                "area_km2": h3.cell_area(cell, unit="km^2"),
                "reference_year": COUNTRY_CONFIG[iso]["reference_year"],
                "survey_label": COUNTRY_CONFIG[iso]["survey_label"],
                "geometry": h3_boundary_polygon(cell),
            })

    cells_gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")

    # centroid decides border cells; one admin1 per H3
    pts = gpd.GeoDataFrame(
        cells_gdf[["country_iso3", "h3_index"]].copy(),
        geometry=gpd.points_from_xy(cells_gdf["centroid_lon"], cells_gdf["centroid_lat"]),
        crs="EPSG:4326",
    )

    # avoid sjoin suffixes
    adm1_join = adm1.rename(columns={"ADM1_NAME": "admin1_name"})[
        ["admin1_name", "geometry"]
    ]

    joined = gpd.sjoin(
        pts,
        adm1_join,
        how="left",
        predicate="within",
    )

    admin = joined.drop(columns=["geometry", "index_right"], errors="ignore")
    admin = admin.drop_duplicates(["country_iso3", "h3_index"])

    cells_gdf = cells_gdf.merge(
        admin,
        on=["country_iso3", "h3_index"],
        how="left",
)

    cols = [
        "h3_index", "country_iso3", "admin1_name", "centroid_lat", "centroid_lon",
        "area_km2", "reference_year", "survey_label", "geometry",
    ]

    cells_gdf = cells_gdf[cols]
    return cells_gdf

if CELLS_PATH.exists():
    cells = gpd.read_parquet(CELLS_PATH)
    print(f"Loaded existing spine: {len(cells):,} rows -> {CELLS_PATH}")
else:
    cells = build_spine()
    cells.to_parquet(CELLS_PATH, compression="snappy", index=False)
    print(f"Wrote spine: {len(cells):,} rows -> {CELLS_PATH}")

print(cells.groupby("country_iso3").size().rename("n_cells").to_string())
cells.head()

Loaded existing spine: 904,494 rows -> h3_tabular_v2/h3_res7_country_spine.parquet
country_iso3
ETH    233094
GHA     60433
GMB      2316
KEN    106744
NGA    190814
SLE     17349
TGO     13537
TZA    158287
ZMB    121920


,h3_index,country_iso3,admin1_name,centroid_lat,centroid_lon,area_km2,reference_year,survey_label,geometry
0,875800480ffffff,NGA,Sokoto,13.721760,6.011540,5.022835,2024,assumed_2024,"POLYGON ((5.99976 13.72764, 6.00113 13.71513, ..."
1,875800481ffffff,NGA,Sokoto,13.722512,6.033741,5.024462,2024,assumed_2024,"POLYGON ((6.02196 13.72839, 6.02332 13.71588, ..."
2,875800482ffffff,NGA,Sokoto,13.702618,6.002494,5.020795,2024,assumed_2024,"POLYGON ((5.99071 13.7085, 5.99208 13.69599, 6..."
3,875800483ffffff,NGA,Sokoto,13.703369,6.024690,5.022421,2024,assumed_2024,"POLYGON ((6.01291 13.70925, 6.01428 13.69674, ..."
4,875800484ffffff,NGA,Sokoto,13.740151,5.998387,5.023247,2024,assumed_2024,"POLYGON ((5.9866 13.74603, 5.98797 13.73352, 5..."


In [5]:
def ee_polygon_feature(row) -> ee.Feature:
    coords = [[float(x), float(y)] for x, y in row.geometry.exterior.coords]
    return ee.Feature(
        ee.Geometry.Polygon([coords], proj="EPSG:4326", geodesic=False),
        {
            "country_iso3": row.country_iso3,
            "h3_index": row.h3_index,
            "reference_year": int(row.reference_year),
        },
    )


def ee_point_feature(row) -> ee.Feature:
    return ee.Feature(
        ee.Geometry.Point([float(row.centroid_lon), float(row.centroid_lat)]),
        {
            "country_iso3": row.country_iso3,
            "h3_index": row.h3_index,
            "reference_year": int(row.reference_year),
        },
    )


def batched_reduce_to_dataframe(
    sub: gpd.GeoDataFrame,
    image: ee.Image,
    reducer: ee.Reducer,
    scale: int,
    geometry_kind: str,
    desc: str,
    tile_scale: int = 4,
) -> pd.DataFrame:
    rows = []
    feature_fn = ee_point_feature if geometry_kind == "point" else ee_polygon_feature

    for start in tqdm(range(0, len(sub), BATCH_SIZE), desc=desc):
        batch = sub.iloc[start:start + BATCH_SIZE]
        fc = ee.FeatureCollection([feature_fn(r) for _, r in batch.iterrows()])

        if geometry_kind == "point":
            sampled = image.sampleRegions(
                collection=fc,
                scale=scale,
                geometries=False,
                tileScale=tile_scale,
            )
        else:
            sampled = image.reduceRegions(
                collection=fc,
                reducer=reducer,
                scale=scale,
                tileScale=tile_scale,
            )

        info = sampled.getInfo()
        rows.extend([f["properties"] for f in info["features"]])

    return pd.DataFrame(rows)


def effective_year(reference_year: int, source_key: str) -> int:
    # survey year wins unless source coverage ends earlier
    return int(min(reference_year, SOURCE_LATEST_YEAR[source_key]))


def country_region_geometry(iso: str) -> ee.Geometry:
    """GAUL country geometry for EE filters."""
    name = COUNTRY_CONFIG[iso]["gaul_name"]
    return ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq("ADM0_NAME", name)).geometry()

## AlphaEarth

Polygon mean per H3 cell. stored as 64 float32 columns, `ae_A00` through `ae_A63`.

In [17]:
AE_COLLECTION = "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
AE_BANDS = [f"A{i:02d}" for i in range(64)]
AE_OUT_BANDS = [f"ae_A{i:02d}" for i in range(64)]

AE_EXPORT_FOLDER = "h3_tabular_v2_ae_exports"
AE_REDUCE_SCALE = 30      # 30 m kept for full-country runtime
AE_CHUNK_SIZE = 3000      # export batch
AE_TILE_SCALE = 8


def alphaearth_image(year: int, region: ee.Geometry) -> ee.Image:
    return (
        ee.ImageCollection(AE_COLLECTION)
        .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
        .filterBounds(region)
        .mosaic()
        .select(AE_BANDS)
        .rename(AE_OUT_BANDS)
        .toFloat()
    )


def start_alphaearth_exports(countries=COUNTRIES, force=False):
    """Start chunked AlphaEarth polygon-mean exports."""
    tasks = {}
    export_dir = DRIVE_ROOT / AE_EXPORT_FOLDER
    export_dir.mkdir(parents=True, exist_ok=True)

    for iso in countries:
        sub_all = cells[cells.country_iso3 == iso].reset_index(drop=True)
        year = effective_year(int(COUNTRY_CONFIG[iso]["reference_year"]), "alphaearth")
        region = country_region_geometry(iso)
        image = alphaearth_image(year, region)

        n_chunks = math.ceil(len(sub_all) / AE_CHUNK_SIZE)
        print(f"{iso}: {len(sub_all):,} cells -> {n_chunks:,} AlphaEarth export chunks")

        for start in range(0, len(sub_all), AE_CHUNK_SIZE):
            sub = sub_all.iloc[start:start + AE_CHUNK_SIZE]
            chunk_id = start // AE_CHUNK_SIZE
            desc = f"alphaearth_{iso}_{year}_s{AE_REDUCE_SCALE}_chunk{chunk_id:04d}"

            existing = sorted(export_dir.glob(f"{desc}*.csv"))
            if existing and not force:
                print(f"  skip existing {desc}")
                continue

            fc = ee.FeatureCollection([
                ee_polygon_feature(r) for _, r in sub.iterrows()
            ])

            # polygon mean, not a centroid sample
            reduced = image.reduceRegions(
                collection=fc,
                reducer=ee.Reducer.mean(),
                scale=AE_REDUCE_SCALE,
                tileScale=AE_TILE_SCALE,
            ).map(
                lambda f: f.set({
                    "alphaearth_effective_year": year,
                    "ae_chunk_id": chunk_id,
                    "ae_reduce_scale": AE_REDUCE_SCALE,
                })
            )

            task = ee.batch.Export.table.toDrive(
                collection=reduced,
                description=desc,
                folder=AE_EXPORT_FOLDER,
                fileNamePrefix=desc,
                fileFormat="CSV",
                selectors=[
                    "country_iso3",
                    "h3_index",
                    "alphaearth_effective_year",
                    "ae_chunk_id",
                    "ae_reduce_scale",
                ] + AE_OUT_BANDS,
            )
            task.start()
            tasks[(iso, chunk_id)] = task
            print(f"  started {desc}: {len(sub):,} cells")

    return tasks


def alphaearth_task_status(tasks):
    rows = []
    for key, task in tasks.items():
        status = task.status()
        rows.append({
            "key": key,
            "state": status.get("state"),
            "description": status.get("description"),
            "error_message": status.get("error_message"),
        })
    return pd.DataFrame(rows)


def load_alphaearth_exports(countries=COUNTRIES) -> pd.DataFrame:
    """Load completed AlphaEarth CSVs and write parquet."""
    export_dir = DRIVE_ROOT / AE_EXPORT_FOLDER
    frames = []

    for iso in countries:
        pattern = f"alphaearth_{iso}_*_s{AE_REDUCE_SCALE}_chunk*.csv"
        matches = sorted(export_dir.glob(pattern))

        if not matches:
            print(f"[WARN] No AlphaEarth export CSVs yet for {iso}: {pattern}")
            continue

        iso_frames = []
        for path in matches:
            df = pd.read_csv(path)
            iso_frames.append(df)

        iso_df = pd.concat(iso_frames, ignore_index=True)
        frames.append(iso_df)
        print(f"{iso}: loaded {len(iso_df):,} rows from {len(matches):,} chunk files")

    if not frames:
        raise FileNotFoundError(
            f"No AlphaEarth export CSVs found in {export_dir}. "
            "Run start_alphaearth_exports() and wait for tasks to complete first."
        )

    ae = pd.concat(frames, ignore_index=True)
    ae = ae.drop_duplicates(["country_iso3", "h3_index"])

    for c in AE_OUT_BANDS:
        if c not in ae.columns:
            ae[c] = np.nan
        ae[c] = pd.to_numeric(ae[c], errors="coerce").astype("float32")

    keep = ["country_iso3", "h3_index", "alphaearth_effective_year"] + AE_OUT_BANDS
    ae = ae[keep]

    ae.to_parquet(ALPHAEARTH_PATH, compression="snappy", index=False)
    print(f"Wrote AlphaEarth parquet: {len(ae):,} rows -> {ALPHAEARTH_PATH}")
    return ae

In [18]:
# start exports
alphaearth_tasks = start_alphaearth_exports()


NGA: 190,814 cells -> 64 AlphaEarth export chunks
  started alphaearth_NGA_2024_s30_chunk0000: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0001: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0002: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0003: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0004: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0005: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0006: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0007: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0008: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0009: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0010: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0011: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0012: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0013: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0014: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0015: 3,000 cells
  started alphaearth_NGA_2024_s30_chun

  started alphaearth_NGA_2024_s30_chunk0028: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0029: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0030: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0031: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0032: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0033: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0034: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0035: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0036: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0037: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0038: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0039: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0040: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0041: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0042: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0043: 3,000 cells
  started alphaearth_NGA_2024_s30_chunk0044: 3,000 cells
  started alphaearth_NGA_2024_s

In [21]:
alphaearth = load_alphaearth_exports()



NGA: loaded 190,814 rows from 64 chunk files
TZA: loaded 158,287 rows from 53 chunk files
ETH: loaded 233,094 rows from 78 chunk files
GMB: loaded 2,316 rows from 1 chunk files
GHA: loaded 60,433 rows from 21 chunk files
KEN: loaded 106,744 rows from 36 chunk files
SLE: loaded 17,349 rows from 6 chunk files
TGO: loaded 13,537 rows from 5 chunk files
ZMB: loaded 121,920 rows from 41 chunk files
Wrote AlphaEarth parquet: 904,494 rows -> h3_tabular_v2/gee_outputs/alphaearth_h3_mean.parquet


In [22]:
# load completed exports
if ALPHAEARTH_PATH.exists():
    alphaearth = pd.read_parquet(ALPHAEARTH_PATH)
    print(f"Loaded AlphaEarth parquet: {len(alphaearth):,} rows -> {ALPHAEARTH_PATH}")
else:
    print("AlphaEarth parquet not found yet.")
    print("Run: alphaearth_tasks = start_alphaearth_exports()")
    print("After tasks complete, run: alphaearth = load_alphaearth_exports()")
    alphaearth = pd.DataFrame(columns=["country_iso3", "h3_index", "alphaearth_effective_year"] + AE_OUT_BANDS)

if len(alphaearth):
    print(alphaearth.groupby("country_iso3")["h3_index"].count().to_string())

alphaearth.head()

Loaded AlphaEarth parquet: 904,494 rows -> h3_tabular_v2/gee_outputs/alphaearth_h3_mean.parquet
country_iso3
ETH    233094
GHA     60433
GMB      2316
KEN    106744
NGA    190814
SLE     17349
TGO     13537
TZA    158287
ZMB    121920


,country_iso3,h3_index,alphaearth_effective_year,ae_A00,ae_A01,ae_A02,ae_A03,ae_A04,ae_A05,ae_A06,ae_A07,ae_A08,ae_A09,ae_A10,ae_A11,ae_A12,ae_A13,ae_A14,ae_A15,ae_A16,ae_A17,ae_A18,ae_A19,ae_A20,ae_A21,ae_A22,ae_A23,ae_A24,ae_A25,ae_A26,ae_A27,ae_A28,ae_A29,ae_A30,ae_A31,ae_A32,ae_A33,ae_A34,ae_A35,ae_A36,ae_A37,ae_A38,ae_A39,ae_A40,ae_A41,ae_A42,ae_A43,ae_A44,ae_A45,ae_A46,ae_A47,ae_A48,ae_A49,ae_A50,ae_A51,ae_A52,ae_A53,ae_A54,ae_A55,ae_A56,ae_A57,ae_A58,ae_A59,ae_A60,ae_A61,ae_A62,ae_A63
0,NGA,875800480ffffff,2024,-0.006053,0.032032,0.252097,0.000063,-0.130114,0.094196,-0.081636,0.085821,-0.163444,0.005057,-0.090926,-0.011422,-0.221832,0.001357,0.064416,0.207339,-0.116695,0.122470,-0.052475,0.140328,-0.097934,-0.152371,0.107460,0.062628,0.015053,-0.057409,0.217076,-0.044949,-0.163176,0.003799,-0.022385,0.034145,-0.135407,-0.037327,-0.073382,-0.152979,-0.213129,0.227619,0.214523,-0.015042,0.193545,-0.028954,-0.135505,0.023096,-0.138388,0.177316,-0.076145,-0.125738,-0.112833,-0.083687,0.137772,0.133608,-0.058292,-0.072246,0.175345,0.001980,0.162393,0.140452,-0.149102,-0.088393,-0.241337,-0.006331,0.061758,0.079357
1,NGA,875800481ffffff,2024,0.003986,0.006210,0.251353,0.017800,-0.113592,0.084060,-0.063910,0.077656,-0.151052,0.021838,-0.066868,-0.012241,-0.225444,0.013862,0.085587,0.195116,-0.137911,0.141399,-0.090502,0.138783,-0.100970,-0.111753,0.107099,0.059950,0.039274,-0.064949,0.230874,-0.047282,-0.144259,0.010964,-0.021668,0.037762,-0.148331,-0.039528,-0.064775,-0.118814,-0.204438,0.228448,0.230603,-0.024743,0.153132,-0.027643,-0.163694,0.031918,-0.114907,0.165624,-0.081699,-0.114760,-0.112524,-0.067964,0.166085,0.145125,-0.073026,-0.091809,0.184505,0.016389,0.139206,0.127529,-0.154171,-0.108459,-0.222500,0.005538,0.056840,0.097856
2,NGA,875800482ffffff,2024,0.013733,0.018365,0.252834,0.015444,-0.124065,0.089801,-0.067780,0.078046,-0.147787,0.002649,-0.073280,-0.008516,-0.219016,0.015263,0.069966,0.215050,-0.116464,0.136143,-0.082744,0.141541,-0.103556,-0.109053,0.127626,0.069645,0.031729,-0.052175,0.228519,-0.043854,-0.161743,0.027843,-0.032017,0.034028,-0.155615,-0.052945,-0.061388,-0.123356,-0.218598,0.224904,0.211794,-0.021005,0.169387,-0.038381,-0.163717,0.036153,-0.120817,0.186749,-0.066350,-0.102573,-0.110883,-0.076045,0.150820,0.140229,-0.064537,-0.085838,0.170798,0.021378,0.137706,0.125693,-0.145932,-0.111998,-0.223593,0.003881,0.062840,0.101187
3,NGA,875800483ffffff,2024,0.025950,0.016367,0.243991,0.017661,-0.116140,0.083922,-0.042587,0.073305,-0.145669,-0.002300,-0.058440,-0.009905,-0.216940,0.044410,0.079258,0.211351,-0.120856,0.155432,-0.114533,0.132006,-0.124510,-0.080812,0.140163,0.067916,0.057404,-0.049064,0.221181,-0.049108,-0.151895,0.037061,-0.036000,0.040260,-0.175068,-0.066962,-0.032429,-0.100387,-0.210923,0.220324,0.206597,-0.018486,0.142446,-0.047323,-0.193630,0.049738,-0.088960,0.187844,-0.073876,-0.079296,-0.110080,-0.053840,0.159005,0.148824,-0.060267,-0.100147,0.175363,0.039602,0.119532,0.108057,-0.163193,-0.139593,-0.195610,-0.004119,0.052928,0.128622
4,NGA,875800484ffffff,2024,-0.004746,0.042343,0.253754,-0.024175,-0.144833,0.102378,-0.082458,0.068770,-0.170185,0.012831,-0.101331,-0.012207,-0.240790,-0.001363,0.065047,0.211730,-0.116517,0.111369,-0.059876,0.139849,-0.070413,-0.159169,0.095017,0.045638,0.000321,-0.088586,0.204214,-0.045650,-0.145259,-0.004251,-0.007736,0.024356,-0.114148,-0.052041,-0.084496,-0.163062,-0.198137,0.218940,0.211248,-0.019496,0.164318,-0.003117,-0.121345,0.027151,-0.139773,0.175098,-0.093797,-0.139986,-0.117161,-0.064096,0.119792,0.128993,-0.053792,-0.076155,0.189345,-0.003010,0.176008,0.141839,-0.131264,-0.066271,-0.239503,-0.028470,0.045014,0.067407


## Nightlights

VIIRS monthly `avg_rad`, 24 months through the effective year.

- raw mean and volatility keep small negative values
- log slope and volatility clamp only before `log1p`
- valid month count kept for QA

In [ ]:
VIIRS_COLLECTION = "NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG"
VIIRS_CF_CVG_MIN = 3  # require 3+ cloud-free looks
VIIRS_LOG_EPS = 1e-6


def month_starts_for_24m(end_year: int) -> list[pd.Timestamp]:
    # 24 complete months
    return list(pd.date_range(f"{end_year - 1}-01-01", f"{end_year}-12-01", freq="MS"))


def viirs_stack(end_year: int, region: ee.Geometry) -> ee.Image:
    bands = []
    for ts in month_starts_for_24m(end_year):
        start = ts.strftime("%Y-%m-%d")
        end = (ts + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")
        name = f"ntl_{ts.year}_{ts.month:02d}"
        img = (
            ee.ImageCollection(VIIRS_COLLECTION)
            .filterDate(start, end)
            .filterBounds(region)
            .select(["avg_rad", "cf_cvg"])
            .median()
        )
        # keep calibrated negatives in raw stats
        rad = img.select("avg_rad").updateMask(img.select("cf_cvg").gte(VIIRS_CF_CVG_MIN)).rename(name)
        bands.append(rad)
    return ee.Image.cat(bands).toFloat()


def summarize_viirs_monthlies(df: pd.DataFrame) -> pd.DataFrame:
    month_cols = [c for c in df.columns if c.startswith("ntl_20")]
    vals = df[month_cols].apply(pd.to_numeric, errors="coerce").astype("float32")

    # clamp only for logs; raw stats stay untouched
    log_input = vals.clip(lower=VIIRS_LOG_EPS)
    log_vals = np.log1p(log_input)
    x = np.arange(len(month_cols), dtype="float32")

    y = log_vals.to_numpy(dtype="float32")
    slopes = []
    for row in y:
        mask = np.isfinite(row)
        if mask.sum() < 6:
            # no trend with fewer than 6 valid months
            slopes.append(np.nan)
        else:
            xx = x[mask]
            yy = row[mask]
            slopes.append(float(np.polyfit(xx, yy, 1)[0]))

    out = df[["country_iso3", "h3_index"]].copy()
    out["ntl_effective_year"] = df["ntl_effective_year"].astype("int16")
    out["ntl_mean_24m"] = vals.mean(axis=1, skipna=True).astype("float32")
    out["ntl_raw_volatility_24m"] = vals.std(axis=1, skipna=True).astype("float32")
    out["ntl_log_slope_24m"] = pd.Series(slopes, dtype="float32")
    out["ntl_log_volatility_24m"] = log_vals.std(axis=1, skipna=True).astype("float32")
    out["ntl_valid_months"] = vals.notna().sum(axis=1).astype("int16")
    return out


def run_viirs() -> pd.DataFrame:
    all_monthly = []
    for iso in COUNTRIES:
        sub = cells[cells.country_iso3 == iso].copy()
        ref_year = int(COUNTRY_CONFIG[iso]["reference_year"])
        year = effective_year(ref_year, "viirs_full_year")
        region = country_region_geometry(iso)
        image = viirs_stack(year, region)
        df = batched_reduce_to_dataframe(
            sub=sub,
            image=image,
            reducer=ee.Reducer.mean(),
            scale=500,
            geometry_kind="polygon",
            desc=f"{iso} VIIRS 24m ending {year}",
            tile_scale=4,
        )
        df["ntl_effective_year"] = year
        all_monthly.append(df)
    monthly = pd.concat(all_monthly, ignore_index=True)
    return summarize_viirs_monthlies(monthly)


if VIIRS_PATH.exists():
    viirs = pd.read_parquet(VIIRS_PATH)
    print(f"Loaded VIIRS: {len(viirs):,}")
else:
    viirs = run_viirs()
    viirs.to_parquet(VIIRS_PATH, compression="snappy", index=False)
    print(f"Wrote VIIRS: {len(viirs):,} -> {VIIRS_PATH}")

## NDVI seasonality

MODIS Terra + Aqua in 23 16-day periods.

- annual mean
- p90-p10 seasonal amplitude
- peak period, day and month
- valid-period count and reliability flags

In [31]:
# Terra + Aqua, 23 x 16-day periods

MODIS_TERRA_COLLECTION = "MODIS/061/MOD13Q1"
MODIS_AQUA_COLLECTION = "MODIS/061/MYD13Q1"

MODIS_FULL_YEAR_MAX = 2025
MODIS_NDVI_SCALE = 250
MODIS_TILE_SCALE = 8
MODIS_CHUNK_SIZE = 3000

MODIS_PERIOD_DAYS = 16
MODIS_N_PERIODS = 23

NDVI_MIN_VALID_PERIODS = 10  # need 10/23 periods for seasonality

NDVI_EXPORT_FOLDER = "h3_tabular_v2_modis_ndvi_exports"
NDVI_EXPORT_TAG = "modis16d_v1"

NDVI_PATH = GEE_OUT / "modis_ndvi_16d_annual.parquet"


def modis_effective_year(ref_year: int) -> int:
    return min(int(ref_year), MODIS_FULL_YEAR_MAX)


def prep_modis_ndvi(img: ee.Image) -> ee.Image:
    raw = img.select("NDVI")
    qa = img.select("SummaryQA")

    # keep QA 0-1; drop snow and cloud
    good = qa.lte(1).And(raw.gte(-2000)).And(raw.lte(10000))

    return (
        raw.multiply(0.0001)
        .updateMask(good)
        .rename("ndvi")
        .copyProperties(img, ["system:time_start"])
    )


def modis_ndvi_collection(year: int, region: ee.Geometry) -> ee.ImageCollection:
    start = f"{year}-01-01"
    end = f"{year + 1}-01-01"

    terra = (
        ee.ImageCollection(MODIS_TERRA_COLLECTION)
        .filterBounds(region)
        .filterDate(start, end)
        .map(prep_modis_ndvi)
    )

    aqua = (
        ee.ImageCollection(MODIS_AQUA_COLLECTION)
        .filterBounds(region)
        .filterDate(start, end)
        .map(prep_modis_ndvi)
    )

    return terra.merge(aqua)


def modis_16d_ndvi_stack(year: int, region: ee.Geometry) -> ee.Image:
    coll = modis_ndvi_collection(year, region)
    year_end = pd.Timestamp(f"{year + 1}-01-01")

    bands = []
    for p in range(MODIS_N_PERIODS):
        start_ts = pd.Timestamp(f"{year}-01-01") + pd.Timedelta(days=MODIS_PERIOD_DAYS * p)
        end_ts = min(start_ts + pd.Timedelta(days=MODIS_PERIOD_DAYS), year_end)

        period_coll = coll.filterDate(
            start_ts.strftime("%Y-%m-%d"),
            end_ts.strftime("%Y-%m-%d"),
        )

        empty = (
            ee.Image.constant(0)
            .updateMask(ee.Image.constant(0))
            .rename(f"ndvi_p{p:02d}")
            .toFloat()
        )

        period_img = ee.Image(
            ee.Algorithms.If(
                period_coll.size().gt(0),
                period_coll.median().rename(f"ndvi_p{p:02d}").toFloat(),
                empty,
            )
        )

        bands.append(period_img)

    return ee.Image.cat(bands).toFloat()


def period_midpoint_lookup(year: int) -> dict[int, tuple[int, int]]:
    out = {}
    for p in range(MODIS_N_PERIODS):
        mid_ts = pd.Timestamp(f"{year}-01-01") + pd.Timedelta(days=MODIS_PERIOD_DAYS * p + MODIS_PERIOD_DAYS // 2)
        if mid_ts.year > year:
            mid_ts = pd.Timestamp(f"{year}-12-31")
        out[p + 1] = (int(mid_ts.dayofyear), int(mid_ts.month))
    return out


def summarize_modis_ndvi_periods(df: pd.DataFrame) -> pd.DataFrame:
    period_cols = [f"ndvi_p{p:02d}" for p in range(MODIS_N_PERIODS)]

    # keep missing periods as NaN
    vals = (
        df.reindex(columns=period_cols)
        .apply(pd.to_numeric, errors="coerce")
        .astype("float32")
    )

    arr = vals.to_numpy(dtype="float32")
    all_nan = ~np.isfinite(arr).any(axis=1)

    peak_period = np.nanargmax(np.where(np.isfinite(arr), arr, -np.inf), axis=1) + 1
    peak_period = peak_period.astype("float32")
    peak_period[all_nan] = np.nan

    valid_periods = vals.notna().sum(axis=1).astype("int16")
    reliable = valid_periods >= NDVI_MIN_VALID_PERIODS

    # p90-p10 avoids single-period spikes
    p90 = vals.quantile(0.90, axis=1, interpolation="linear")
    p10 = vals.quantile(0.10, axis=1, interpolation="linear")
    amplitude = (p90 - p10).astype("float32")
    amplitude = amplitude.where(reliable, np.nan)

    out = df[["country_iso3", "h3_index"]].copy()
    out["ndvi_effective_year"] = pd.to_numeric(df["ndvi_effective_year"], errors="coerce").astype("Int16")
    out["ndvi_mean"] = vals.mean(axis=1, skipna=True).astype("float32")
    out["ndvi_seasonal_amplitude"] = amplitude.astype("float32")
    out["ndvi_amplitude_reliable"] = reliable.astype(bool)
    out["ndvi_peak_period"] = pd.Series(peak_period, dtype="float32").astype("Int16")
    out["ndvi_peak_period_reliable"] = reliable.astype(bool)
    out["ndvi_valid_periods"] = valid_periods

    # peak timing from period midpoint
    peak_doy = []
    peak_month = []
    cache = {}

    for year, pp in zip(out["ndvi_effective_year"], out["ndvi_peak_period"]):
        if pd.isna(year) or pd.isna(pp):
            peak_doy.append(np.nan)
            peak_month.append(np.nan)
            continue

        year = int(year)
        pp = int(pp)
        if year not in cache:
            cache[year] = period_midpoint_lookup(year)

        doy, month = cache[year][pp]
        peak_doy.append(doy)
        peak_month.append(month)

    out["ndvi_peak_doy"] = pd.Series(peak_doy, dtype="float32").astype("Int16")
    out["ndvi_peak_month"] = pd.Series(peak_month, dtype="float32").astype("Int16")
    out["ndvi_peak_month_reliable"] = reliable.astype(bool)

    return out


def chunk_dataframe(df: pd.DataFrame, chunk_size: int):
    for start in range(0, len(df), chunk_size):
        yield start // chunk_size, df.iloc[start:start + chunk_size].copy()


def start_modis_ndvi_exports(countries=COUNTRIES):
    tasks = {}

    for iso in countries:
        sub = cells[cells.country_iso3 == iso].copy().reset_index(drop=True)
        ref_year = int(COUNTRY_CONFIG[iso]["reference_year"])
        year = modis_effective_year(ref_year)
        region = country_region_geometry(iso)

        image = modis_16d_ndvi_stack(year, region)

        for chunk_id, chunk in chunk_dataframe(sub, MODIS_CHUNK_SIZE):
            fc = ee.FeatureCollection([ee_polygon_feature(r) for _, r in chunk.iterrows()])

            reduced = image.reduceRegions(
                collection=fc,
                reducer=ee.Reducer.mean(),
                scale=MODIS_NDVI_SCALE,
                tileScale=MODIS_TILE_SCALE,
            ).map(lambda f: f.set("ndvi_effective_year", year))

            desc = f"modis_ndvi_{NDVI_EXPORT_TAG}_{iso}_{year}_s{MODIS_NDVI_SCALE}_chunk{chunk_id:05d}"

            task = ee.batch.Export.table.toDrive(
                collection=reduced,
                description=desc,
                folder=NDVI_EXPORT_FOLDER,
                fileNamePrefix=desc,
                fileFormat="CSV",
                selectors=[
                    "country_iso3",
                    "h3_index",
                    "ndvi_effective_year",
                    *[f"ndvi_p{p:02d}" for p in range(MODIS_N_PERIODS)],
                ],
            )

            task.start()
            tasks[(iso, chunk_id)] = task
            print(f"Started {desc}: {len(chunk):,} cells")

    return tasks


def modis_ndvi_task_status(tasks):
    rows = []
    for key, task in tasks.items():
        s = task.status()
        rows.append({
            "key": key,
            "state": s.get("state"),
            "description": s.get("description"),
            "error_message": s.get("error_message"),
        })
    return pd.DataFrame(rows)


def load_modis_ndvi_exports() -> pd.DataFrame:
    export_dir = DRIVE_ROOT / NDVI_EXPORT_FOLDER
    frames = []

    for iso in COUNTRIES:
        pattern = f"modis_ndvi_{NDVI_EXPORT_TAG}_{iso}_*_s{MODIS_NDVI_SCALE}_chunk*.csv"
        files = sorted(export_dir.glob(pattern))

        if not files:
            print(f"[WARN] No MODIS NDVI export CSVs found for {iso}: {pattern}")
            continue

        iso_frames = []
        for path in files:
            df = pd.read_csv(path)
            iso_frames.append(df)

        iso_df = pd.concat(iso_frames, ignore_index=True)
        iso_df = iso_df.drop_duplicates(["country_iso3", "h3_index"])
        frames.append(iso_df)

        print(f"{iso}: loaded {len(iso_df):,} rows from {len(files):,} chunk files")

    if not frames:
        raise FileNotFoundError(f"No MODIS NDVI export CSVs found under {export_dir}")

    periods = pd.concat(frames, ignore_index=True)
    ndvi = summarize_modis_ndvi_periods(periods)
    return ndvi




In [32]:
# start exports
modis_ndvi_tasks = start_modis_ndvi_exports()
display(modis_ndvi_task_status(modis_ndvi_tasks))


Started modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00000: 3,000 cells
Started modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00001: 3,000 cells
Started modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00002: 3,000 cells
Started modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00003: 3,000 cells
Started modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00004: 3,000 cells
Started modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00005: 3,000 cells
Started modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00006: 3,000 cells
Started modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00007: 3,000 cells
Started modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00008: 3,000 cells
Started modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00009: 3,000 cells
Started modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00010: 3,000 cells
Started modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00011: 3,000 cells
Started modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00012: 3,000 cells
Started modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00013: 3,000 cells
Started modis_ndvi_modis16d_v1_NGA

Started modis_ndvi_modis16d_v1_ETH_2025_s250_chunk00025: 3,000 cells
Started modis_ndvi_modis16d_v1_ETH_2025_s250_chunk00026: 3,000 cells
Started modis_ndvi_modis16d_v1_ETH_2025_s250_chunk00027: 3,000 cells
Started modis_ndvi_modis16d_v1_ETH_2025_s250_chunk00028: 3,000 cells
Started modis_ndvi_modis16d_v1_ETH_2025_s250_chunk00029: 3,000 cells
Started modis_ndvi_modis16d_v1_ETH_2025_s250_chunk00030: 3,000 cells
Started modis_ndvi_modis16d_v1_ETH_2025_s250_chunk00031: 3,000 cells
Started modis_ndvi_modis16d_v1_ETH_2025_s250_chunk00032: 3,000 cells
Started modis_ndvi_modis16d_v1_ETH_2025_s250_chunk00033: 3,000 cells
Started modis_ndvi_modis16d_v1_ETH_2025_s250_chunk00034: 3,000 cells
Started modis_ndvi_modis16d_v1_ETH_2025_s250_chunk00035: 3,000 cells
Started modis_ndvi_modis16d_v1_ETH_2025_s250_chunk00036: 3,000 cells
Started modis_ndvi_modis16d_v1_ETH_2025_s250_chunk00037: 3,000 cells
Started modis_ndvi_modis16d_v1_ETH_2025_s250_chunk00038: 3,000 cells
Started modis_ndvi_modis16d_v1_ETH

,key,state,description,error_message
0,"(NGA, 0)",COMPLETED,modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00000,None
1,"(NGA, 1)",COMPLETED,modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00001,None
2,"(NGA, 2)",COMPLETED,modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00002,None
3,"(NGA, 3)",COMPLETED,modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00003,None
4,"(NGA, 4)",COMPLETED,modis_ndvi_modis16d_v1_NGA_2024_s250_chunk00004,None
...,...,...,...,...
300,"(ZMB, 36)",COMPLETED,modis_ndvi_modis16d_v1_ZMB_2024_s250_chunk00036,None
301,"(ZMB, 37)",COMPLETED,modis_ndvi_modis16d_v1_ZMB_2024_s250_chunk00037,None
302,"(ZMB, 38)",COMPLETED,modis_ndvi_modis16d_v1_ZMB_2024_s250_chunk00038,None
303,"(ZMB, 39)",COMPLETED,modis_ndvi_modis16d_v1_ZMB_2024_s250_chunk00039,None


In [33]:

# load completed exports
ndvi = load_modis_ndvi_exports()
ndvi.to_parquet(NDVI_PATH, compression="snappy", index=False)
print(f"Wrote MODIS 16-day NDVI: {len(ndvi):,} -> {NDVI_PATH}")
ndvi.head()

NGA: loaded 190,814 rows from 64 chunk files
TZA: loaded 158,287 rows from 53 chunk files
ETH: loaded 233,094 rows from 78 chunk files
GMB: loaded 2,316 rows from 1 chunk files
GHA: loaded 60,433 rows from 21 chunk files
KEN: loaded 106,744 rows from 36 chunk files
SLE: loaded 17,349 rows from 6 chunk files
TGO: loaded 13,537 rows from 5 chunk files
ZMB: loaded 121,920 rows from 41 chunk files
Wrote MODIS 16-day NDVI: 904,494 -> h3_tabular_v2/gee_outputs/modis_ndvi_16d_annual.parquet


,country_iso3,h3_index,ndvi_effective_year,ndvi_mean,ndvi_seasonal_amplitude,ndvi_amplitude_reliable,ndvi_peak_period,ndvi_peak_period_reliable,ndvi_valid_periods,ndvi_peak_doy,ndvi_peak_month,ndvi_peak_month_reliable
0,NGA,875800480ffffff,2024,0.236040,0.272050,True,17,True,20,265,9,True
1,NGA,875800481ffffff,2024,0.252173,0.250121,True,17,True,20,265,9,True
2,NGA,875800482ffffff,2024,0.252541,0.272081,True,17,True,20,265,9,True
3,NGA,875800483ffffff,2024,0.289283,0.292471,True,15,True,21,233,8,True
4,NGA,875800484ffffff,2024,0.227258,0.223591,True,17,True,23,265,9,True


## GHSL height

Static 2018 building height: mean, p90 and max per cell.

In [42]:
GHSL_HEIGHT_YEAR = 2018
GHSL_HEIGHT_EXPORT_FOLDER = "h3_tabular_v2_ghsl_height_exports"
GHSL_HEIGHT_EXPORT_TAG = "ghslh_v1"

GHSL_HEIGHT_SCALE = 100
GHSL_HEIGHT_TILE_SCALE = 8
GHSL_HEIGHT_CHUNK_SIZE = 5000

GHSL_HEIGHT_RAW_PATH = GEE_OUT / "ghsl_building_height_raw.parquet"


def ghsl_height_image() -> ee.Image:
    return (
        ee.Image(f"JRC/GHSL/P2023A/GHS_BUILT_H/{GHSL_HEIGHT_YEAR}")
        .select("built_height")
        .rename("ghsl_built_height_m")
        .toFloat()
    )


def chunk_dataframe(df: pd.DataFrame, chunk_size: int):
    for start in range(0, len(df), chunk_size):
        yield start // chunk_size, df.iloc[start:start + chunk_size].copy()


def start_ghsl_height_exports(countries=COUNTRIES):
    tasks = {}
    image = ghsl_height_image()

    # mean + p90 + max kept for vertical form
    reducer = (
        ee.Reducer.mean()
        .combine(ee.Reducer.percentile([90]), sharedInputs=True)
        .combine(ee.Reducer.max(), sharedInputs=True)
    )

    for iso in countries:
        sub = cells[cells.country_iso3 == iso].copy().reset_index(drop=True)

        for chunk_id, chunk in chunk_dataframe(sub, GHSL_HEIGHT_CHUNK_SIZE):
            fc = ee.FeatureCollection([ee_polygon_feature(r) for _, r in chunk.iterrows()])

            reduced = image.reduceRegions(
                collection=fc,
                reducer=reducer,
                scale=GHSL_HEIGHT_SCALE,
                tileScale=GHSL_HEIGHT_TILE_SCALE,
            ).map(lambda f: f.set("ghsl_height_effective_year", GHSL_HEIGHT_YEAR))

            desc = f"ghsl_height_{GHSL_HEIGHT_EXPORT_TAG}_{iso}_{GHSL_HEIGHT_YEAR}_s{GHSL_HEIGHT_SCALE}_chunk{chunk_id:05d}"

            task = ee.batch.Export.table.toDrive(
                collection=reduced,
                description=desc,
                folder=GHSL_HEIGHT_EXPORT_FOLDER,
                fileNamePrefix=desc,
                fileFormat="CSV",
                selectors=[
                    "country_iso3",
                    "h3_index",
                    "ghsl_height_effective_year",
                    "ghsl_built_height_m",
                    "ghsl_built_height_m_mean",
                    "ghsl_built_height_m_p90",
                    "ghsl_built_height_m_max",
                ],
            )

            task.start()
            tasks[(iso, chunk_id)] = task
            print(f"Started {desc}: {len(chunk):,} cells")

    return tasks


def ghsl_height_task_status(tasks):
    rows = []
    for key, task in tasks.items():
        s = task.status()
        rows.append({
            "key": key,
            "state": s.get("state"),
            "description": s.get("description"),
            "error_message": s.get("error_message"),
        })
    return pd.DataFrame(rows)


def load_ghsl_height_exports() -> pd.DataFrame:
    export_dir = DRIVE_ROOT / GHSL_HEIGHT_EXPORT_FOLDER
    frames = []

    for iso in COUNTRIES:
        pattern = f"ghsl_height_{GHSL_HEIGHT_EXPORT_TAG}_{iso}_{GHSL_HEIGHT_YEAR}_s{GHSL_HEIGHT_SCALE}_chunk*.csv"
        files = sorted(export_dir.glob(pattern))

        if not files:
            print(f"[WARN] No GHSL height CSVs found for {iso}: {pattern}")
            continue

        iso_df = pd.concat([pd.read_csv(p) for p in files], ignore_index=True)
        iso_df = iso_df.drop_duplicates(["country_iso3", "h3_index"])
        frames.append(iso_df)

        print(f"{iso}: loaded {len(iso_df):,} rows from {len(files):,} chunk files")

    if not frames:
        raise FileNotFoundError(f"No GHSL height export CSVs found under {export_dir}")

    raw = pd.concat(frames, ignore_index=True)
    raw.to_parquet(GHSL_HEIGHT_RAW_PATH, compression="snappy", index=False)
    print(f"Wrote raw GHSL height: {len(raw):,} -> {GHSL_HEIGHT_RAW_PATH}")

    return raw


def postprocess_ghsl_height(raw: pd.DataFrame) -> pd.DataFrame:
    raw = raw.copy()

    print("Raw GHSL columns:")
    print(raw.columns.tolist())

    # reducer names vary by EE export
    if "ghsl_built_height_m_mean" in raw.columns:
        raw["ghsl_built_height_mean_m"] = raw["ghsl_built_height_m_mean"]
    elif "ghsl_built_height_m" in raw.columns:
        raw["ghsl_built_height_mean_m"] = raw["ghsl_built_height_m"]
    elif "mean" in raw.columns:
        raw["ghsl_built_height_mean_m"] = raw["mean"]
    else:
        raise KeyError(f"No GHSL mean column found. Available: {raw.columns.tolist()}")

    if "ghsl_built_height_m_p90" in raw.columns:
        raw["ghsl_built_height_p90_m"] = raw["ghsl_built_height_m_p90"]
    elif "ghsl_built_height_m_p90_1" in raw.columns:
        raw["ghsl_built_height_p90_m"] = raw["ghsl_built_height_m_p90_1"]
    elif "p90" in raw.columns:
        raw["ghsl_built_height_p90_m"] = raw["p90"]
    else:
        raise KeyError(f"No GHSL p90 column found. Available: {raw.columns.tolist()}")

    if "ghsl_built_height_m_max" in raw.columns:
        raw["ghsl_built_height_max_m"] = raw["ghsl_built_height_m_max"]
    elif "max" in raw.columns:
        raw["ghsl_built_height_max_m"] = raw["max"]
    else:
        raise KeyError(f"No GHSL max column found. Available: {raw.columns.tolist()}")

    expected = [
        "ghsl_built_height_mean_m",
        "ghsl_built_height_p90_m",
        "ghsl_built_height_max_m",
    ]

    for c in expected:
        raw[c] = pd.to_numeric(raw[c], errors="coerce").astype("float32")

    raw["ghsl_height_effective_year"] = 2018

    keep = [
        "country_iso3",
        "h3_index",
        "ghsl_height_effective_year",
        "ghsl_built_height_mean_m",
        "ghsl_built_height_p90_m",
        "ghsl_built_height_max_m",
    ]

    return raw[keep].copy()

In [43]:
# start exports
ghsl_height_tasks = start_ghsl_height_exports()
display(ghsl_height_task_status(ghsl_height_tasks))

Started ghsl_height_ghslh_v1_NGA_2018_s100_chunk00000: 5,000 cells
Started ghsl_height_ghslh_v1_NGA_2018_s100_chunk00001: 5,000 cells
Started ghsl_height_ghslh_v1_NGA_2018_s100_chunk00002: 5,000 cells
Started ghsl_height_ghslh_v1_NGA_2018_s100_chunk00003: 5,000 cells
Started ghsl_height_ghslh_v1_NGA_2018_s100_chunk00004: 5,000 cells
Started ghsl_height_ghslh_v1_NGA_2018_s100_chunk00005: 5,000 cells
Started ghsl_height_ghslh_v1_NGA_2018_s100_chunk00006: 5,000 cells
Started ghsl_height_ghslh_v1_NGA_2018_s100_chunk00007: 5,000 cells
Started ghsl_height_ghslh_v1_NGA_2018_s100_chunk00008: 5,000 cells
Started ghsl_height_ghslh_v1_NGA_2018_s100_chunk00009: 5,000 cells
Started ghsl_height_ghslh_v1_NGA_2018_s100_chunk00010: 5,000 cells
Started ghsl_height_ghslh_v1_NGA_2018_s100_chunk00011: 5,000 cells
Started ghsl_height_ghslh_v1_NGA_2018_s100_chunk00012: 5,000 cells
Started ghsl_height_ghslh_v1_NGA_2018_s100_chunk00013: 5,000 cells
Started ghsl_height_ghslh_v1_NGA_2018_s100_chunk00014: 5,000 c

Started ghsl_height_ghslh_v1_ETH_2018_s100_chunk00045: 5,000 cells
Started ghsl_height_ghslh_v1_ETH_2018_s100_chunk00046: 3,094 cells
Started ghsl_height_ghslh_v1_GMB_2018_s100_chunk00000: 2,316 cells
Started ghsl_height_ghslh_v1_GHA_2018_s100_chunk00000: 5,000 cells
Started ghsl_height_ghslh_v1_GHA_2018_s100_chunk00001: 5,000 cells
Started ghsl_height_ghslh_v1_GHA_2018_s100_chunk00002: 5,000 cells
Started ghsl_height_ghslh_v1_GHA_2018_s100_chunk00003: 5,000 cells
Started ghsl_height_ghslh_v1_GHA_2018_s100_chunk00004: 5,000 cells
Started ghsl_height_ghslh_v1_GHA_2018_s100_chunk00005: 5,000 cells
Started ghsl_height_ghslh_v1_GHA_2018_s100_chunk00006: 5,000 cells
Started ghsl_height_ghslh_v1_GHA_2018_s100_chunk00007: 5,000 cells
Started ghsl_height_ghslh_v1_GHA_2018_s100_chunk00008: 5,000 cells
Started ghsl_height_ghslh_v1_GHA_2018_s100_chunk00009: 5,000 cells
Started ghsl_height_ghslh_v1_GHA_2018_s100_chunk00010: 5,000 cells
Started ghsl_height_ghslh_v1_GHA_2018_s100_chunk00011: 5,000 c

,key,state,description,error_message
0,"(NGA, 0)",COMPLETED,ghsl_height_ghslh_v1_NGA_2018_s100_chunk00000,None
1,"(NGA, 1)",COMPLETED,ghsl_height_ghslh_v1_NGA_2018_s100_chunk00001,None
2,"(NGA, 2)",COMPLETED,ghsl_height_ghslh_v1_NGA_2018_s100_chunk00002,None
3,"(NGA, 3)",COMPLETED,ghsl_height_ghslh_v1_NGA_2018_s100_chunk00003,None
4,"(NGA, 4)",COMPLETED,ghsl_height_ghslh_v1_NGA_2018_s100_chunk00004,None
...,...,...,...,...
181,"(ZMB, 20)",COMPLETED,ghsl_height_ghslh_v1_ZMB_2018_s100_chunk00020,None
182,"(ZMB, 21)",COMPLETED,ghsl_height_ghslh_v1_ZMB_2018_s100_chunk00021,None
183,"(ZMB, 22)",COMPLETED,ghsl_height_ghslh_v1_ZMB_2018_s100_chunk00022,None
184,"(ZMB, 23)",COMPLETED,ghsl_height_ghslh_v1_ZMB_2018_s100_chunk00023,None


In [46]:
raw_ghsl_height = pd.read_parquet(GHSL_HEIGHT_RAW_PATH)



Raw GHSL columns:
['country_iso3', 'h3_index', 'ghsl_height_effective_year', 'ghsl_built_height_m', 'ghsl_built_height_m_mean', 'ghsl_built_height_m_p90', 'ghsl_built_height_m_max']
Wrote GHSL height: 904,494 -> h3_tabular_v2/gee_outputs/ghsl_building_height.parquet


,country_iso3,h3_index,ghsl_height_effective_year,ghsl_built_height_mean_m,ghsl_built_height_p90_m,ghsl_built_height_max_m
0,NGA,875800480ffffff,2018,NaN,NaN,NaN
1,NGA,875800481ffffff,2018,NaN,NaN,NaN
2,NGA,875800482ffffff,2018,NaN,NaN,NaN
3,NGA,875800483ffffff,2018,NaN,NaN,NaN
4,NGA,875800484ffffff,2018,NaN,NaN,NaN


In [44]:
# load completed exports
raw_ghsl_height = load_ghsl_height_exports()

ghsl_height = postprocess_ghsl_height(raw_ghsl_height)
ghsl_height.to_parquet(GHSL_HEIGHT_PATH, compression="snappy", index=False)

print(f"Wrote GHSL height: {len(ghsl_height):,} -> {GHSL_HEIGHT_PATH}")
ghsl_height.head()

NGA: loaded 190,814 rows from 39 chunk files
TZA: loaded 158,287 rows from 32 chunk files
ETH: loaded 233,094 rows from 47 chunk files
GMB: loaded 2,316 rows from 1 chunk files
GHA: loaded 60,433 rows from 13 chunk files
KEN: loaded 106,744 rows from 22 chunk files
SLE: loaded 17,349 rows from 4 chunk files
TGO: loaded 13,537 rows from 3 chunk files
ZMB: loaded 121,920 rows from 25 chunk files
Wrote raw GHSL height: 904,494 -> h3_tabular_v2/gee_outputs/ghsl_building_height_raw.parquet
Raw GHSL columns:
['country_iso3', 'h3_index', 'ghsl_height_effective_year', 'ghsl_built_height_m', 'ghsl_built_height_m_mean', 'ghsl_built_height_m_p90', 'ghsl_built_height_m_max']


## GEE core merge

AlphaEarth, VIIRS, NDVI and GHSL joined to the H3 spine.

In [11]:
# reload after kernel reset

alphaearth = pd.read_parquet(ALPHAEARTH_PATH)
viirs = pd.read_parquet(VIIRS_PATH)
ndvi = pd.read_parquet(NDVI_PATH)
ghsl_height = pd.read_parquet(GHSL_HEIGHT_PATH)

print(f"alphaearth:  {len(alphaearth):,} rows, {len(alphaearth.columns):,} cols -> {ALPHAEARTH_PATH}")
print(f"viirs:       {len(viirs):,} rows, {len(viirs.columns):,} cols -> {VIIRS_PATH}")
print(f"ndvi:        {len(ndvi):,} rows, {len(ndvi.columns):,} cols -> {NDVI_PATH}")
print(f"ghsl_height: {len(ghsl_height):,} rows, {len(ghsl_height.columns):,} cols -> {GHSL_HEIGHT_PATH}")

for name, df in [
    ("alphaearth", alphaearth),
    ("viirs", viirs),
    ("ndvi", ndvi),
    ("ghsl_height", ghsl_height),
]:
    dup = df.duplicated(["country_iso3", "h3_index"]).sum()
    print(f"{name} duplicate keys: {dup:,}")
    if dup:
        raise ValueError(f"{name} has duplicate country_iso3/h3_index")

alphaearth:  904,494 rows, 67 cols -> h3_tabular_v2/gee_outputs/alphaearth_h3_mean.parquet
viirs:       904,494 rows, 8 cols -> h3_tabular_v2/gee_outputs/viirs_nightlights_24m_unfloored.parquet
ndvi:        904,494 rows, 12 cols -> h3_tabular_v2/gee_outputs/modis_ndvi_16d_annual.parquet
ghsl_height: 904,494 rows, 6 cols -> h3_tabular_v2/gee_outputs/ghsl_building_height.parquet
alphaearth duplicate keys: 0
viirs duplicate keys: 0
ndvi duplicate keys: 0
ghsl_height duplicate keys: 0


In [12]:
def merge_one_to_one(left, right, label):
    before = len(left)
    out = left.merge(right, on=["country_iso3", "h3_index"], how="left", validate="one_to_one")
    assert len(out) == before, f"{label} changed row count"
    return out


gee_features = cells.drop(columns=["geometry"]).copy()
for frame, label in [
    (alphaearth, "alphaearth"),
    (viirs, "viirs"),
    (ndvi, "ndvi"),
    (ghsl_height, "ghsl_height"),
]:
    gee_features = merge_one_to_one(gee_features, frame, label)

gee_features.to_parquet(GEE_FEATURES_PATH, compression="snappy", index=False)
print(f"Wrote GEE core features: {len(gee_features):,} rows, {len(gee_features.columns):,} columns -> {GEE_FEATURES_PATH}")
gee_features.groupby("country_iso3").size().rename("n").to_string()

Wrote GEE core features: 904,494 rows, 93 columns -> h3_tabular_v2/gee_outputs/gee_h3_features.parquet


'country_iso3\nETH    233094\nGHA     60433\nGMB      2316\nKEN    106744\nNGA    190814\nSLE     17349\nTGO     13537\nTZA    158287\nZMB    121920'

## Google Open Buildings

Direct S2-token downloads by country, then H3 morphology:

- count, density and footprint share
- area mean, spread and gini
- nearest-neighbor regularity
- orientation entropy

In [7]:
# GOB direct pull by S2 token
# output csv.gz keeps the source columns
!pip -q install s2geometry geopandas

import functools, glob, gzip, multiprocessing, os, shutil, tempfile
from pathlib import Path
from typing import List, Optional, Tuple

import geopandas as gpd
import pandas as pd
import shapely
import s2geometry as s2
import tensorflow as tf
import tqdm.notebook

GOB_CONFIDENCE_MIN = 0.70          # same cutoff across countries
DATA_TYPE = "polygons"
BUILDING_DOWNLOAD_PATH = f"gs://open-buildings-data/v3/{DATA_TYPE}_s2_level_6_gzip_no_header"

GOB_DOWNLOAD_DIR = OUTPUT_ROOT / "gob_direct_downloads"
GOB_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# NGA/TZA use existing local files; ETH/TGO dropped
GOB_DIRECT_COUNTRIES = ["GMB", "GHA", "KEN", "SLE", "ZMB"]
OVERWRITE = False


def get_filename_and_region_dataframe(iso: str) -> Tuple[str, gpd.GeoDataFrame]:
    url = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip"
    shp = os.path.basename(url)
    if not os.path.exists(shp):
        os.system(f"wget -q -N {url}")
    filename = f"open_buildings_v3_{DATA_TYPE}_ne_10m_{iso}.csv.gz"
    region_df = (
        gpd.read_file(shp).query(f'ISO_A3 == "{iso}"')
        .dissolve(by="ISO_A3")[["geometry"]]
    )
    return filename, region_df


def get_bounding_box_s2_covering_tokens(region_geometry) -> List[str]:
    b = region_geometry.bounds
    rect = s2.S2LatLngRect.FromPointPair(
        s2.S2LatLng.FromDegrees(b[1], b[0]), s2.S2LatLng.FromDegrees(b[3], b[2]))
    coverer = s2.S2RegionCoverer()
    coverer.set_fixed_level(6)            # source tiles are level 6
    coverer.set_max_cells(1000000)
    return [c.ToToken() for c in coverer.GetCovering(rect)]


def s2_token_to_shapely_polygon(s2_token: str):
    cell = s2.S2Cell(s2.S2CellId.FromToken(s2_token))
    coords = []
    for i in range(4):
        ll = s2.S2LatLng(cell.GetVertex(i))
        coords.append((ll.lng().degrees(), ll.lat().degrees()))
    return shapely.geometry.Polygon(coords)


def download_s2_token(s2_token: str, region_df: gpd.GeoDataFrame) -> Optional[str]:
    cell_geom = s2_token_to_shapely_polygon(s2_token)
    region_geom = region_df.iloc[0].geometry
    prepared = shapely.prepared.prep(region_geom)
    if not prepared.intersects(cell_geom):
        return None
    try:
        with tf.io.gfile.GFile(
            os.path.join(BUILDING_DOWNLOAD_PATH, f"{s2_token}_buildings.csv.gz"), "rb") as gf:
            if prepared.covers(cell_geom):          # no point filter needed
                with tempfile.NamedTemporaryFile(mode="w+b", delete=False) as tmp:
                    shutil.copyfileobj(gf, tmp)
                    return tmp.name
            chunks = pd.read_csv(gf, chunksize=2_000_000, dtype=object,
                                 compression="gzip", header=None)
            tmp = tempfile.NamedTemporaryFile(mode="w+b", delete=False); tmp.close()
            for ch in chunks:
                pts = gpd.GeoDataFrame(
                    geometry=gpd.points_from_xy(ch[1], ch[0]), crs="EPSG:4326")
                pts = gpd.sjoin(pts, region_df, predicate="within")
                ch.iloc[pts.index].to_csv(
                    tmp.name, mode="ab", index=False, header=False,
                    compression={"method": "gzip", "compresslevel": 1})
            return tmp.name
    except tf.errors.NotFoundError:
        return None


def download_gob_country(iso: str) -> Path:
    filename, region_df = get_filename_and_region_dataframe(iso)
    out_path = GOB_DOWNLOAD_DIR / filename
    if out_path.exists() and not OVERWRITE:
        print(f"[SKIP] {iso}: exists -> {out_path} ({out_path.stat().st_size/1e9:.2f} GB)")
        return out_path

    temp_root = Path(tempfile.gettempdir())
    for f in temp_root.glob("open_buildings_*"):
        f.unlink()
    temp_file = temp_root / filename
    with gzip.open(temp_file, "wt") as merged:   # header once
        merged.write(",".join(
            ["latitude", "longitude", "area_in_meters",
             "confidence", "geometry", "full_plus_code"]) + "\n")

    fn = functools.partial(download_s2_token, region_df=region_df)
    tokens = get_bounding_box_s2_covering_tokens(region_df.iloc[0].geometry)
    print(f"{iso}: {len(tokens):,} S2 level-6 tokens -> {filename}")
    with open(temp_file, "ab") as merged:        # concat gzip members
        with multiprocessing.Pool(4) as pool:
            for fname in tqdm.notebook.tqdm(
                pool.imap_unordered(fn, tokens), total=len(tokens), desc=f"{iso} GOB"):
                if fname:
                    with open(fname, "rb") as t:
                        shutil.copyfileobj(t, merged)
                    os.unlink(fname)

    shutil.copy2(temp_file, out_path)
    print(f"{iso}: {out_path} ({out_path.stat().st_size/1e9:.2f} GB)")
    return out_path


# skip existing files
gob_files = {iso: download_gob_country(iso) for iso in GOB_DIRECT_COUNTRIES}
print("\nDone.")
for iso, p in gob_files.items():
    print(f"{iso}: {p} ({p.stat().st_size/1e9:.2f} GB)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 51.1 MB/s eta 0:00:00


In [11]:
from tqdm.auto import tqdm as tqdm_bar   # separate from tqdm.notebook

GOB_CHUNK_ROWS = 1_000_000
ORIENTATION_BINS = 18
COMPUTE_ORIENTATION = True   # geometry parse is the slow part

# active direct-download countries
GOB_PROCESS_COUNTRIES = ["GMB", "GHA", "KEN", "SLE", "ZMB"]


def gini(values) -> float:
    values = np.asarray(values, dtype="float64")
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    if np.any(values < 0):
        values = values - np.nanmin(values)
    total = values.sum()
    if total <= 0:
        return np.nan
    values = np.sort(values)
    n = values.size
    return float((2 * np.arange(1, n + 1) @ values) / (n * total) - (n + 1) / n)


def normalized_entropy(angles, bins: int = ORIENTATION_BINS) -> float:
    angles = np.asarray(angles, dtype="float64")
    angles = angles[np.isfinite(angles)]
    if angles.size < 3:
        return np.nan
    hist, _ = np.histogram(angles % 180.0, bins=bins, range=(0, 180))
    p = hist[hist > 0] / hist.sum()
    return float(-(p * np.log(p)).sum() / np.log(bins))


def building_orientations_deg(wkt_values) -> np.ndarray:
    """Longest MRR edge angle, vectorized."""
    geoms = shapely.from_wkt(np.asarray(wkt_values, dtype=object), on_invalid="ignore")
    rect = shapely.oriented_envelope(geoms)                 # batched MRR
    coords, idx = shapely.get_coordinates(rect, return_index=True)
    out = np.full(len(geoms), np.nan)
    if coords.shape[0] < 2:
        return out
    same = idx[:-1] == idx[1:]
    d = coords[1:][same] - coords[:-1][same]
    gi = idx[:-1][same]
    seg_len = np.hypot(d[:, 0], d[:, 1])
    seg_ang = np.degrees(np.arctan2(d[:, 1], d[:, 0])) % 180.0
    order = np.lexsort((seg_len, gi))                       # longest edge sorts last
    gis, angs = gi[order], seg_ang[order]
    last = np.empty(gis.shape[0], dtype=bool)
    last[-1] = True
    last[:-1] = gis[1:] != gis[:-1]                         # one angle per geometry
    out[gis[last]] = angs[last]
    return out


def compute_gob_morphology_for_country(iso: str) -> pd.DataFrame:
    path = GOB_DOWNLOAD_DIR / f"open_buildings_v3_polygons_ne_10m_{iso}.csv.gz"
    if not path.exists():
        print(f"[WARN] No GOB file for {iso}: {path}")
        return pd.DataFrame(columns=["country_iso3", "h3_index"])

    usecols = ["latitude", "longitude", "area_in_meters", "confidence"]
    if COMPUTE_ORIENTATION:
        usecols.append("geometry")
    dtypes = {"latitude": "float64", "longitude": "float64",
              "area_in_meters": "float64", "confidence": "float64"}

    records = []
    print(f"{iso}: reading {path.name}")
    for chunk in pd.read_csv(path, chunksize=GOB_CHUNK_ROWS, compression="gzip",
                             usecols=usecols, dtype=dtypes):
        chunk = chunk[chunk["confidence"] >= GOB_CONFIDENCE_MIN]
        if chunk.empty:
            continue
        lat = chunk["latitude"].to_numpy()
        lon = chunk["longitude"].to_numpy()
        h3idx = [h3.latlng_to_cell(float(a), float(o), H3_RES) for a, o in zip(lat, lon)]
        orient = (building_orientations_deg(chunk["geometry"].to_numpy())
                  if COMPUTE_ORIENTATION else np.full(len(chunk), np.nan))
        records.append(pd.DataFrame({
            "h3_index": h3idx,
            "area_in_meters": chunk["area_in_meters"].to_numpy(),
            "centroid_lat": lat,
            "centroid_lon": lon,
            "_orientation": orient,
        }))

    if not records:
        return pd.DataFrame(columns=["country_iso3", "h3_index"])

    b = pd.concat(records, ignore_index=True)

    rows = []
    for h, g in tqdm_bar(b.groupby("h3_index", sort=False), desc=f"{iso} GOB cells"):
        n = len(g)
        area = g["area_in_meters"].to_numpy()
        nn_cv = np.nan
        if n >= 3:  # spacing CV needs at least 3 buildings
            lat = np.deg2rad(g["centroid_lat"].to_numpy())
            lon = np.deg2rad(g["centroid_lon"].to_numpy())
            x = 6371000.0 * lon * np.cos(np.nanmean(lat))
            y = 6371000.0 * lat
            pts = np.column_stack([x, y])
            dists, _ = cKDTree(pts).query(pts, k=2)
            nn = dists[:, 1]
            mean_nn = np.nanmean(nn)
            if mean_nn > 0:
                nn_cv = float(np.nanstd(nn) / mean_nn)
        rows.append({
            "country_iso3": iso,
            "h3_index": h,
            "gob_building_count": int(n),
            "gob_building_area_total_m2": float(np.nansum(area)),
            "gob_building_area_mean_m2": float(np.nanmean(area)) if n else np.nan,
            "gob_building_area_std_m2": float(np.nanstd(area)) if n else np.nan,
            "gob_building_size_gini": gini(area),
            "gob_nn_cv": nn_cv,
            "gob_spacing_regularity": float(1.0 / (1.0 + nn_cv)) if np.isfinite(nn_cv) else np.nan,
            "gob_orientation_entropy": normalized_entropy(g["_orientation"].to_numpy()),
            "gob_effective_year": SOURCE_LATEST_YEAR["gob_inference"],
        })
    return pd.DataFrame(rows)


def run_gob_morphology() -> pd.DataFrame:
    frames = [compute_gob_morphology_for_country(iso) for iso in GOB_PROCESS_COUNTRIES]
    gob = pd.concat(frames, ignore_index=True)
    spine = cells.drop(columns=["geometry"])
    spine = spine[spine["country_iso3"].isin(GOB_PROCESS_COUNTRIES)][["country_iso3", "h3_index", "area_km2"]]
    gob = spine.merge(gob, on=["country_iso3", "h3_index"], how="left", validate="one_to_one")
    gob["gob_building_count"] = gob["gob_building_count"].fillna(0).astype("int32")
    gob["gob_building_area_total_m2"] = gob["gob_building_area_total_m2"].fillna(0).astype("float32")
    gob["gob_building_density_per_km2"] = (gob["gob_building_count"] / gob["area_km2"]).astype("float32")
    gob["gob_footprint_fraction"] = (
        gob["gob_building_area_total_m2"] / (gob["area_km2"] * 1_000_000.0)
    ).clip(0, 1).astype("float32")
    gob["gob_effective_year"] = gob["gob_effective_year"].fillna(SOURCE_LATEST_YEAR["gob_inference"]).astype("int16")
    return gob.drop(columns=["area_km2"])




In [ ]:
gob_morph = run_gob_morphology()
gob_morph.to_parquet(GOB_MORPH_PATH, compression="snappy", index=False)

## Colab-side merge

Legacy merge path. disabled.

In [ ]:
RWI_FEATURES_PATH = OUTPUT_ROOT / "local_outputs" / "rwi_h3_features.parquet"
DHS_FEATURES_PATH = OUTPUT_ROOT / "local_outputs" / "dhs_h3_features.parquet"
WORLDPOP_FEATURES_PATH = OUTPUT_ROOT / "local_outputs" / "worldpop_h3_features.parquet"
OSM_FEATURES_PATH = OUTPUT_ROOT / "local_outputs" / "osm_h3_features.parquet"


def optional_read_parquet(path: Path, label: str) -> pd.DataFrame | None:
    if path.exists():
        df = pd.read_parquet(path)
        print(f"Loaded {label}: {len(df):,} rows, {len(df.columns):,} columns")
        return df
    print(f"[WARN] Missing {label}: {path}")
    return None


RUN_OPTIONAL_COLAB_MERGE = False

if not RUN_OPTIONAL_COLAB_MERGE:
    print("Colab-side merge disabled")
else:
    final = pd.read_parquet(GEE_FEATURES_PATH)

    for path, label in [
        (GOB_MORPH_PATH, "GOB morphology"),
        (RWI_FEATURES_PATH, "RWI local features"),
        (DHS_FEATURES_PATH, "DHS local features"),
        (WORLDPOP_FEATURES_PATH, "WorldPop local features"),
        (OSM_FEATURES_PATH, "OSM local features"),
    ]:
        df = optional_read_parquet(path, label)
        if df is not None:
            before = len(final)
            final = final.merge(df, on=["country_iso3", "h3_index"], how="left", validate="one_to_one")
            assert len(final) == before, f"{label} changed row count"

    # keep AE columns float32
    for c in [c for c in final.columns if c.startswith("ae_A")]:
        final[c] = pd.to_numeric(final[c], errors="coerce").astype("float32")

    final.to_parquet(FINAL_PATH, compression="snappy", index=False)